정수계획 모형

In [2]:
from ortools.linear_solver import pywraplp

# Create the mip solver with the SCIP backend.
solver = pywraplp.Solver.CreateSolver("SAT")

infinity = solver.infinity()
# x and y are integer non-negative variables.
# x = solver.IntVar(0.0,infinity, "x")
# y = solver.IntVar(0.0,infinity, "y")
x = solver.NumVar(0.0,infinity, "x")
y = solver.NumVar(0.0,infinity, "y")

solver.Add(x + 7*y <=17.5)
solver.Add(x<=3.5)

solver.Maximize(x+10*y)

print(f"Solving with {solver.SolverVersion()}")
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Solution:")
    print("Objective value=", solver.Objective().Value())
    print("x=", x.solution_value())
    print("y=", y.solution_value())
else:
    print("The problem does not have an optimal solution.")


Solving with CP-SAT solver v9.15.6755
Solution:
Objective value= 25.0
x= 0.0
y= 2.5


### 예제1. 스태프 결정 문제 - Code

x[i] : i 요일에 일을 시작하는 스태프의 수

In [14]:
from ortools.linear_solver import pywraplp

# Create the mip solver with the SCIP backend.
solver = pywraplp.Solver.CreateSolver("SAT")

REQ = [20,16,13,16,19,14,12]
names = ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']

infinity = solver.infinity()

x = {}
for i in range(7):
    x[i] = solver.IntVar(0,infinity, "x[%i]" %i)

for i in range(7):
    constraint_expr = [x[(i-j)%7] for j in range(5)]
    solver.Add(sum(constraint_expr)>=REQ[i], names[i])

objective = solver.Objective()
solver.Minimize(solver.Sum(x[i] for i in range(7)))

# 모델 파일 생성
with open('or5-1-1.lp', "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Objective value=", solver.Objective().Value())
    for i in range(7):
        print(x[i].name(), "=", x[i].solution_value())
else:
    print("The problem does not have an optimal solution.")

print()
print("각 요일에 근무하는 종업원 수 : ")
for i in range(7):
    print(names[i], str(sum([x[(i-j)%7].solution_value() for j in range(5)])))


Objective value= 22.0
x[0] = 8.0
x[1] = 2.0
x[2] = 0.0
x[3] = 6.0
x[4] = 3.0
x[5] = 3.0
x[6] = 0.0

각 요일에 근무하는 종업원 수 : 
MON 20.0
TUE 16.0
WED 13.0
THU 16.0
FRI 19.0
SAT 14.0
SUN 12.0


### 예제2. 팀배정 문제

X[i,j] : 1 if 컨설턴트 i와 j가 한쌍  
R_i,j : 분석가 i와 j의 팀워크 점수

In [15]:
from ortools.linear_solver import pywraplp

# Create the mip solver with the SCIP backend.
solver = pywraplp.Solver.CreateSolver("SAT")

Ratings = [
    [None, 9, 3, 4, 2, 1, 5, 6],
    [None, None, 1, 7, 3, 5, 2, 1],
    [None, None, None, 4, 4, 2, 9, 2],
    [None, None, None, None, 1, 5, 5, 2],
    [None, None, None, None, None, 8, 7, 6],
    [None, None, None, None, None, None, 2, 3],
    [None, None, None, None, None, None, None, 4]
]

nC = len(Ratings[0]) # Ratings[0] : [None, 9, 3, 4, 2, 1, 5, 6]

# 의사결정 변수
X = {}
for i in range(nC):
    for j in range(nC):
        if i !=j:
            X[i,j] = solver.IntVar(0,1,"X"+str(i)+str(j))

# 제약조건
const_expr = []

# 컨설턴트당 1명만 배정
for i in range(nC):
    const_expr = [X[i,j] for j in range(nC) if i != j]
    # const_expr : [X01, X02, X03, X04, X05, X06, X07]
    solver.Add(sum(const_expr) == 1, 'consultant_' + str(i))

# xij = xji 제약
for i in range(nC):
    for j in range(nC):
        if i < j:
            solver.Add(X[i, j] == X[j, i], 'x_' + str(i) + '_' + str(j))

# 목적함수
obj_expr = []

for i in range(nC-1):
    for j in range(nC):
        if Ratings[i][j] != None:
            obj_expr.append(Ratings[i][j] * X[i, j])

solver.Maximize(solver.Sum(obj_expr))

# 모델 파일 생성
with open("or5-1-2.lp", "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Objective value = %.1f" % solver.Objective().Value())
    for i in range(nC-1):
        for j in range(nC):
            if i != j and X[i, j].solution_value() != 0:
                if Ratings[i][j] != None:
                    # %i는 정수를 출력하기 위한 포맷 문자
                    print("Consultant [%i]" %(i+1), " --> Consultant [%i]" %(j+1),
                        "팀워크 점수 : %i" %(Ratings[i][j]))
else:
    print("The problem does not have an optimal solution.")

Objective value = 30.0
Consultant [1]  --> Consultant [8] 팀워크 점수 : 6
Consultant [2]  --> Consultant [4] 팀워크 점수 : 7
Consultant [3]  --> Consultant [7] 팀워크 점수 : 9
Consultant [5]  --> Consultant [6] 팀워크 점수 : 8


### 예제3. 정수 계획 문제

- 대구, 광주 중 하나 또는 둘에 공장을 세운다.(yes or no)
- 종속 조건 "공장이 A에 세워지면 -> 창고도 A에 세운다"  
- 창고는 1개  
    
위를 만족하기 위해,  
- x3+x4<=1
- x3<=x1
- x4<=x2  
(x1: 대구공장, x2: 광주공장, x3: 대구창고, x4: 광주창고)

In [ ]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")

nVars = 4
values = [9, 5, 6, 4]
reqs = [6, 3, 5, 2]

# 의사결정변수
x = {}
for i in range(nVars):
    x[i] = solver.IntVar(0, 1, "x[%i]" % i)

# 제약조건
solver.Add(sum([reqs[i]*x[i] for i in range(nVars)]) <= 11, 'const 0')
solver.Add(x[2] + x[3] <= 1, 'const 1')
solver.Add(x[2] <= x[0], 'const 2')
solver.Add(x[3] <= x[1], 'const 3')

# 목적함수
solver.Maximize(sum([values[i]*x[i] for i in range(nVars)]))

with open('or5-1-3.lp', "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("OPTIMAL")
    print("목적함수값 = ", solver.Objective().Value())
    for i in range(nVars):
        print(x[i].name(), " = ", x[i].solution_value())
else:
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수값 =  18.0
x[0]  =  1.0
x[1]  =  1.0
x[2]  =  0.0
x[3]  =  1.0


결론 : 대구, 광주에 둘다 공장을 세우나, 창고는 광주에만 세운다.

### 예제4. 정수 계획 문제 - 고정비용 문제
- 교수님이 문제가 이상하다고 인정해주심
- 고친다면: 한곳의 공장에서만 생산하려고 한다

BigM(가능한 최대 생산량보다 충분히 큰 값) 도입하는 이유:  
**공장1,2의 제약 중 어떤 제약을 켜고 끌지를 설정할 수 있게 해줌.**  
앞의 문제와는 달리 공장 1,2에 대한 개별속성 제약식이 있음  
  
이때,  
x1: 장난감1 생산량  
x2: 장난감2 생산량  
y1: 장난감1 생산 여부(0 or 1)  
y2: 장난감2 생산 여부(0 or 1)  
  
이를 통해 생산 여부(yn)가 생산량(xn)을 통제할 수 있도록 한다.

In [7]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")
infinity = solver.infinity()
BIGM = 1000000

# 의사결정변수
x = {}
for i in range(2):
    x[i] = solver.IntVar(0, infinity, "x[%i]" % i)

y = {}
for i in range(3):
    y[i] = solver.IntVar(0, 1, "y[%i]" % i)

# 제약조건
solver.Add(x[0]/50 + x[1]/40 <= 500 + BIGM * y[0], 'const 0')
solver.Add(x[0]/40 + x[1]/25 <= 700 + BIGM * (1 - y[0]), 'const 1')
solver.Add(x[0] <= BIGM * y[1], 'const 2')
solver.Add(x[1] <= BIGM * y[2], 'const 3')

# 목적함수
solver.Maximize(10 * x[0] + 15 * x[1] - 50000 * y[1] - 80000 * y[2])

with open('or5-1.lp', "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("OPTIMAL")
    print("목적함수값 = ", solver.Objective().Value())
    for i in range(2):
        print(x[i].name(), " = ", x[i].solution_value())
    for i in range(3):
        print(y[i].name(), " = ", y[i].solution_value())
else:
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수값 =  230000.0
x[0]  =  28000.0
x[1]  =  0.0
y[0]  =  1.0
y[1]  =  1.0
y[2]  =  0.0


결론
- x[0] (=장난감1) : 28000개  
- x[1] (=장난감2) : 0개  
- y[0] : 제약 2 활성화(=공장 2를 가동)  
- y[1] : 장난감1 고정비 o  
- y[2] : 장난감2 고정비 x  

실습1. 스태프 결정 문제 2

**x[i]는 시간대 i에 시작하는 인원수**

In [1]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SAT")

REQ = [2,2,2,2,2,2,8,8,8,8,
       4,4,3,3,3,3,6,6,5,5,
       5,5,3,3]

infinity = solver.infinity()

# 의사결정변수: i시에 일을 시작하는 근무자 수
x = {}
for i in range(24):
    x[i] = solver.IntVar(0, infinity, "x[%i]" % i)

# 제약조건: 각 시간대 수요 충족
# 시간 (t)의 수요는
# (t, t-1, t-2, ~~ t-8) 시점에 시작한 사람들로 충족된다.
for t in range(24):
    constraint_expr = [
        x[t],
        x[(t-1) % 24],
        x[(t-2) % 24],
        x[(t-3) % 24],
        x[(t-5) % 24],
        x[(t-6) % 24],
        x[(t-7) % 24],
        x[(t-8) % 24]
    ]
    solver.Add(sum(constraint_expr) >= REQ[t], "cover[%i]" % t)

# 목적함수: 총 고용 인원 최소화
solver.Minimize(solver.Sum(x[i] for i in range(24)))

# 모델 파일 생성
with open('or5-2-1.lp', "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Objective value =", solver.Objective().Value())
    for i in range(24):
        print("%i시"%(0+i), " : ", x[i].solution_value())
else:
    print("The problem does not have an optimal solution.")

Objective value = 16.0
0시  :  0.0
1시  :  1.0
2시  :  0.0
3시  :  1.0
4시  :  1.0
5시  :  1.0
6시  :  4.0
7시  :  1.0
8시  :  0.0
9시  :  0.0
10시  :  0.0
11시  :  1.0
12시  :  0.0
13시  :  0.0
14시  :  2.0
15시  :  2.0
16시  :  1.0
17시  :  0.0
18시  :  1.0
19시  :  0.0
20시  :  0.0
21시  :  0.0
22시  :  0.0
23시  :  0.0


실습2. 컨설턴스 배정 최적화

In [ ]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")

Ratings = [
    [15, 18, 21, 12, 25, 19, 14, 22],
    [20, 14, 17, 23, 16, 21, 18, 13],
    [11, 22, 19, 16, 20, 15, 23, 17],
    [24, 16, 13, 20, 18, 22, 11, 19],
    [17, 20, 22, 14, 13, 17, 20, 16],
    [19, 13, 16, 21, 22, 14, 17, 20],
    [22, 17, 20, 18, 15, 20, 16, 21]
]

nConsultants = len(Ratings)       # 7
nProjects = len(Ratings[0])       # 8

# 의사결정변수
X = {}
for i in range(nConsultants):
    for j in range(nProjects):
        X[i, j] = solver.IntVar(0, 1, "X[%i,%i]" % (i, j))

# 제약조건 1: 각 컨설턴트는 정확히 1개의 프로젝트에 배정
for i in range(nConsultants):
    solver.Add(sum(X[i, j] for j in range(nProjects)) == 1, "consultant_%i" % i)

# 제약조건 2: 각 프로젝트는 최대 1명의 컨설턴트만 배정
for j in range(nProjects):
    solver.Add(sum(X[i, j] for i in range(nConsultants)) <= 1, "project_%i" % j)

# 목적함수: 총 배정 비용 최소화
obj_expr = []
for i in range(nConsultants):
    for j in range(nProjects):
        obj_expr.append(Ratings[i][j] * X[i, j])

solver.Minimize(solver.Sum(obj_expr))

# 모델 파일 생성
with open("or5-2-2.lp", "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Objective value = %.1f" % solver.Objective().Value())
    for i in range(nConsultants):
        for j in range(nProjects):
            if X[i, j].solution_value() == 1:
                print("Consultant [%i] --> Project [%i]" % (i + 1, j + 1))
else:
    print("The problem does not have an optimal solution.")

Objective value = 91.0
Consultant [1] --> Project [4]
Consultant [2] --> Project [8]
Consultant [3] --> Project [1]
Consultant [4] --> Project [3]
Consultant [5] --> Project [5]
Consultant [6] --> Project [2]
Consultant [7] --> Project [7]


실습3. 생산계획 문제1

1. "가용한 생산 용량 중 생산에 필요한 용량(주문 당)"    
A사가 생산 가능한 전체 용량에서 소모되는 용량을 지칭  
  
2. 요청이 종류가 2개고, 각 요청별로 주문량이 상이함.  
-> 변수 종류 두종류 있어야. 이진 하나, 정수변수 하나  
(x는 i번째 고객의 제품 생산량, y는 i번째 고객의 제품 생산 여부(이진 변수), M은 bigM  
  
3. y는 해당 고객을 선택했는지 여부를 지칭한다. 이때, 이건 고정비가 있을 때만 필요하다.  

In [2]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")
infinity = solver.infinity()
BIGM = 100000000

# 의사결정변수
x = {}
for i in range(3):
    x[i] = solver.IntVar(0, infinity, "x[%i]" % i)

y = {}
for i in range(2):
    y[i] = solver.IntVar(0, 1, "y[%i]" % i)

# 제약조건
# 전체 생산량
solver.Add(0.2 * x[0] + 0.4 * x[1] + 0.2 * x[2] <= 1, 'const 0')
solver.Add(x[0] <= BIGM * y[0], 'const 1')
solver.Add(x[1] <= BIGM * y[1], 'const 2')
solver.Add(x[0] <= 3, 'const 3')
solver.Add(x[1] <= 2, 'const 4')
solver.Add(x[2] <= 5, 'const 5')

# 목적함수
solver.Maximize(2 * x[0] + 3 * x[1] + 0.8 * x[2] - 3 * y[0] - 2 * y[1])

# 모델 파일 생성
with open('or5-2-3.lp', "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("OPTIMAL")
    print("목적함수값 = ", solver.Objective().Value())
    for i in range(3):
        print(x[i].name(), " = ", x[i].solution_value())
    for i in range(2):
        print(y[i].name(), " = ", y[i].solution_value())
else:
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수값 =  4.800000000000001
x[0]  =  0.0
x[1]  =  2.0
x[2]  =  1.0
y[0]  =  0.0
y[1]  =  1.0


실습4. 생산계획 문제2

1. 생산량 xn, 생산여부 yn, 이때, y5는 마지막 either 용  
  
2. y3+y4<=y1+y2말고 (y3<=y1+y2) & (y4<=y1+y2)를 써야 하는 이유:  
1이나 2 중 하나만 생산해도 3과 4 둘다 생산 가능해야하지만,  
첫 제약식은 둘 중 하나만 생산하면 3,4를 동시에 못 만들게 막기에
  
3. 두 제약식 중 하나를 선택하는 상황. x5를 만들고 bigM을 도입해야  
  
4. bigM은 x1~x4 각각에도 쓰여야하지만, x1~x4 조합해서 사용되는 either 식에도 2번 쓰여야한다. M*y5 & M*(1-y5)

In [4]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")
infinity = solver.infinity()
BIGM = 100000000

# 의사결정변수
x = {}
for i in range(4):
    x[i] = solver.IntVar(0, infinity, "x[%i]" % i)

y = {}
for i in range(5):
    y[i] = solver.IntVar(0, 1, "y[%i]" % i)

# 제약조건
solver.Add(y[0] + y[1] + y[2] + y[3] <= 2, 'const 0')
solver.Add(y[2] <= y[0] + y[1], 'const 1')
solver.Add(y[3] <= y[0] + y[1], 'const 2')
solver.Add(5 * x[0] + 3 * x[1] + 6 * x[2] + 4 * x[3] <= 6000 + BIGM * y[4], 'const 3')
solver.Add(4 * x[0] + 6 * x[1] + 3 * x[2] + 5 * x[3] <= 6000 + BIGM * (1 - y[4]), 'const 4')
solver.Add(x[0] <= BIGM * y[0], 'const 5')
solver.Add(x[1] <= BIGM * y[1], 'const 6')
solver.Add(x[2] <= BIGM * y[2], 'const 7')
solver.Add(x[3] <= BIGM * y[3], 'const 8')

# 목적함수
solver.Maximize(
    70 * x[0] - 50000 * y[0]
    + 60 * x[1] - 40000 * y[1]
    + 90 * x[2] - 70000 * y[2]
    + 80 * x[3] - 60000 * y[3]
)

# 모델 파일 생성
with open('or5-2-4.lp', "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("OPTIMAL")
    print("목적함수값 = ", solver.Objective().Value())
    for i in range(4):
        print("Product %i" %(i+1), " : ", x[i].solution_value())
    for i in range(5):
        if i != 4:
            print("Product %i 생산 여부" %(i+1), " : ", y[i].solution_value())
        else:
            print("제품별 사용량 선택(0:1번식 선택, 1: 2번식 선택)", " : ", y[i].solution_value())
else:
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수값 =  80000.0
Product 1  :  0.0
Product 2  :  2000.0
Product 3  :  0.0
Product 4  :  0.0
Product 1 생산 여부  :  0.0
Product 2 생산 여부  :  1.0
Product 3 생산 여부  :  0.0
Product 4 생산 여부  :  0.0
제품별 사용량 선택(0:1번식 선택, 1: 2번식 선택)  :  0.0


실습5. 할당문제

0. 순서 : 소방서 설치-> 소방서의 담당 배정
1. 구역 소방서 설치 여부 :  x[0]+...x[4] = 2
2. 특정 구역 담당 여부(j구역 담당 소방서) : y[i][0]+...y[i][0]=1, ..., y[i][4]+y[i][4]=1
    (i: 소방서, j: 담당 구역)
3. (중요)소방서의 배치 보장 : y[0][0]<=x[0], y[0][1] <=x[0],....,y[4][3]<=x[4], y[4][4]<=x[4]  
(=소방서 없는 구역값으로 계산되면 안됨)

In [5]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")

# 대응시간(분)
T = [
    [5, 12, 30, 20, 15],
    [20, 4, 15, 10, 25],
    [15, 20, 6, 15, 12],
    [25, 15, 25, 4, 10],
    [10, 25, 15, 12, 5]
]

# 1일 평균 화재 횟수
F = [2, 1, 3, 1, 3]

n = 5

# 의사결정변수
# x[i] = 구역 i에 소방서를 설치하면 1, 아니면 0
x = {}
for i in range(n):
    x[i] = solver.IntVar(0, 1, "x[%i]" % i)

# y[i,j] = 구역 j의 화재를 구역 i의 소방서가 담당하면 1, 아니면 0
y = {}
for i in range(n):
    for j in range(n):
        y[i, j] = solver.IntVar(0, 1, "y[%i][%i]" % (i, j))

# 제약조건
# 소방서는 정확히 2개 설치
solver.Add(sum(x[i] for i in range(n)) == 2, "two_station")

# 각 구역의 화재는 정확히 1개의 소방서에 의해 담당
for j in range(n):
    solver.Add(sum(y[i, j] for i in range(n)) == 1, "assign_st_%i" % j)

# 소방서가 설치된 구역만 담당 가능
# 방지하고 싶은 상황 : 설치 안된 구역 값으로 계산하는것.
for i in range(n):
    for j in range(n):
        solver.Add(y[i, j] <= x[i], "st배치보장_%i_%i" % (i, j))

# 목적함수: 평균 화재 횟수를 가중치로 둔 총 대응시간 최소화
# 평균 화재 횟수 * 대응 시간 * 화재 담당 여부
obj_expr = []
for i in range(n):
    for j in range(n):
        obj_expr.append(F[j] * T[i][j] * y[i, j])

solver.Minimize(solver.Sum(obj_expr))

# 모델 파일 생성
with open("or5-2-5.lp", "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("OPTIMAL")
    print("목적함수값 = ", solver.Objective().Value())

    print("\n[소방서 설치 구역]")
    for i in range(n):
        print("구역 %i (0: 미설치, 1: 설치)"%(i+1), " : ", x[i].solution_value())

    print("\n[구역별 담당 소방서]")
    for j in range(n):
        for i in range(n):
            if y[i, j].solution_value() == 1:
                print("구역 [%i] --> 소방서 [%i]" % (j + 1, i + 1))
else:
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수값 =  85.0

[소방서 설치 구역]
구역 1 (0: 미설치, 1: 설치)  :  0.0
구역 2 (0: 미설치, 1: 설치)  :  0.0
구역 3 (0: 미설치, 1: 설치)  :  1.0
구역 4 (0: 미설치, 1: 설치)  :  0.0
구역 5 (0: 미설치, 1: 설치)  :  1.0

[구역별 담당 소방서]
구역 [1] --> 소방서 [5]
구역 [2] --> 소방서 [3]
구역 [3] --> 소방서 [3]
구역 [4] --> 소방서 [5]
구역 [5] --> 소방서 [5]
